In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tabulate import tabulate

# Path to dataset
dataset_path = '/content/drive/MyDrive/archive/'

# Step 1: Collect labels from folders
labels = []
for folder in os.listdir(dataset_path):
    folder_path = os.path.join(dataset_path, folder)
    if os.path.isdir(folder_path):
        for file in os.listdir(folder_path):
            if file.endswith(".wav"):
                labels.append(folder)  # folder name is the emotion label

# Step 2: Convert to numpy array
labels = np.array(labels)

# Step 3: Encode labels
le = LabelEncoder()
encoded_labels = le.fit_transform(labels)

# Step 4: Create table
table = [[emotion, np.sum(encoded_labels == idx)] for idx, emotion in enumerate(le.classes_)]
print(tabulate(table, headers=['Emotion', 'Number of Samples'], tablefmt='grid'))


In [ ]:
# How to get the emotion label from the audio file name in the RAVDESS dataset

import os

filename = "03-01-05-01-01-01-01.wav"           #MM-PP-EE-LL-II-SS-CC.wav
emotion_code = filename.split('-')[2]  # '05'

emotion_map = {
    '01':'neutral', '02':'calm', '03':'happy', '04':'sad',
    '05':'angry', '06':'fearful', '07':'disgust', '08':'surprised'
}

emotion_label = emotion_map[emotion_code]
print(emotion_label)


In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import os

# Step 3: Path to your dataset folder in Drive
dataset_path = '/content/drive/MyDrive/archive/'  # Your folder

# Find the first .wav file in the dataset_path for demonstration
# This assumes there is at least one subfolder with .wav files
audio_file_path = None
for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.endswith(".wav"):
            audio_file_path = os.path.join(root, file)
            break
    if audio_file_path:
        break

if audio_file_path:
    # 1. Load the audio file
    y, sr = librosa.load(audio_file_path, duration=3, offset=0.5)

    # 2. Extract MFCC features (e.g., n_mfcc=40 as in your code)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)

    # 3. Visualize the MFCCs
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(mfccs, x_axis='time', sr=sr)
    plt.colorbar(format='%+2.0f dB')
    plt.title('MFCC Spectrogram')
    plt.tight_layout()
    plt.show()

    # The next line is how you get the feature vector for your model:
    mfccs_scaled = np.mean(mfccs.T, axis=0)
    print(f"Shape of the final feature vector: {mfccs_scaled.shape}")
else:
    print(f"No .wav files found in the specified dataset path: {dataset_path}")

In [ ]:
## ----------LOGISTIC REGRESSION-------------

# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Step 2: Import libraries
import os
import numpy as np
import librosa
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Step 3: Path to your dataset folder in Drive
dataset_path = '/content/drive/MyDrive/archive/archive/'  # Your folder

# Step 4: Function to extract features from audio files
def extract_features(file_path):
    y, sr = librosa.load(file_path, duration=3, offset=0.5)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    mfccs_scaled = np.mean(mfccs.T, axis=0)
    return mfccs_scaled

# Step 5: Load dataset and extract features
X = []
y = []

for folder in os.listdir(dataset_path):
    folder_path = os.path.join(dataset_path, folder)
    if os.path.isdir(folder_path):
        for file in os.listdir(folder_path):
            if file.endswith(".wav"):
                file_path = os.path.join(folder_path, file)
                features = extract_features(file_path)
                X.append(features)
                y.append(folder)  # folder name is emotion label

X = np.array(X)
y = np.array(y)

# Step 6: Encode labels and scale features
le = LabelEncoder()
y = le.fit_transform(y)

scaler = StandardScaler()
X = scaler.fit_transform(X)

# Step 7: Split dataset into training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 8: Logistic Regression model
model = LogisticRegression(max_iter=500)
model.fit(X_train, y_train)

# Step 9: Predict and evaluate
y_pred = model.predict(X_test)
print("Logistic Regression = ")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))


In [ ]:
##----------NAIVE BAYES-----------------

# Step 8: Naive Bayes model
from sklearn.naive_bayes import GaussianNB

model_nb = GaussianNB()
model_nb.fit(X_train, y_train)

# Step 9: Predict and evaluate
y_pred_nb = model_nb.predict(X_test)

print("Naive Bayes = ")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb, target_names=le.classes_))


In [ ]:
##---------KNN-------------

# Import KNN
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# Train KNN
# You can start with k=5; you can tune it later
model_knn = KNeighborsClassifier(n_neighbors=5)
model_knn.fit(X_train, y_train)

# Predict on test set
y_pred_knn = model_knn.predict(X_test)

# Evaluate
print("KNN Accuracy:", accuracy_score(y_test, y_pred_knn))
print(classification_report(y_test, y_pred_knn, target_names=le.classes_))

In [ ]:
##----------KNN 2D PLOTTING-----------

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier

# Train KNN
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

# Reduce features to 2D using PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_train)

# Plot training data in 2D
plt.figure(figsize=(8,6))
for label in np.unique(y_train):
    plt.scatter(X_pca[y_train == label, 0], X_pca[y_train == label, 1], label=le.classes_[label])
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.title('KNN Training Data (PCA 2D)')
plt.legend()
plt.show()

In [ ]:
##-----------DECISION TREE------------

# Import Decision Tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# Train Decision Tree
model_dt = DecisionTreeClassifier(random_state=42)
model_dt.fit(X_train, y_train)

# Predict on test set
y_pred_dt = model_dt.predict(X_test)

# Evaluate
print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt, target_names=le.classes_))

In [ ]:
##-----------SVM------------

# Import SVM
from sklearn.svm import SVC

# X_train, X_test, y_train, y_test are assumed to be ready from your MFCC/audio features
# Default SVM
model_svm = SVC()  # default rbf kernel
model_svm.fit(X_train, y_train)
print("SVM Accuracy (default):", model_svm.score(X_test, y_test))

# Predict a sample (optional)
# Example: model_svm.predict([X_test[0]])
model_svm_C = SVC(C=5)
model_svm_C.fit(X_train, y_train)
print("SVM Accuracy (C=5):", model_svm_C.score(X_test, y_test))

# Tune Gamma
model_svm_gamma = SVC(gamma=0.01)
model_svm_gamma.fit(X_train, y_train)
print("SVM Accuracy (gamma=0.01):", model_svm_gamma.score(X_test, y_test))

# Linear Kernel
model_svm_linear = SVC(kernel='linear')
model_svm_linear.fit(X_train, y_train)
print("SVM Accuracy (linear kernel):", model_svm_linear.score(X_test, y_test))

In [ ]:
##------plotting of SVM---------
import matplotlib.pyplot as plt

# Store all accuracies
accuracies = {
    "Default RBF": model_svm.score(X_test, y_test),
    "C = 5 (RBF)": model_svm_C.score(X_test, y_test),
    "Gamma = 0.01": model_svm_gamma.score(X_test, y_test),
    "Linear Kernel": model_svm_linear.score(X_test, y_test)
}

# Plot
plt.figure(figsize=(8,5))
plt.bar(accuracies.keys(), accuracies.values())
plt.xlabel("SVM Models")
plt.ylabel("Accuracy")
plt.title("Comparison of SVM Models")
plt.ylim(0,1)  # accuracy range
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(rotation=20)
plt.show()

In [ ]:
##---------confusion matrix---------
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Predict on test data
y_pred = model_svm.predict(X_test)

# Create confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=set(y_test))
disp.plot(cmap='Greens')
plt.title("Confusion Matrix - SVM (Default RBF Kernel)")
plt.show()

In [ ]:
#------plotting for all accuracies-------

import matplotlib.pyplot as plt

# Data
models = ["Logistic Regression", "KNN", "Naive Bayes", "Decision Tree", "SVM"]
accuracy = [95, 81, 68, 53, 96]

# Plot
plt.figure(figsize=(10, 6))
plt.bar(models, accuracy)
plt.xlabel("Models")
plt.ylabel("Accuracy (%)")
plt.title("Model Accuracy Comparison")
plt.ylim(0, 100)
plt.grid(axis='y')
plt.xticks(rotation=20)

plt.show()

# **Record voice,Upload file in drive,detect emotion**

In [ ]:
!pip install -q transformers torchaudio librosa soundfile

import os
import torch
import torchaudio
import torch.nn.functional as F
from google.colab import drive
from transformers import AutoFeatureExtractor, Wav2Vec2ForSequenceClassification

drive.mount('/content/drive')

AUDIO_FILE = "/content/drive/MyDrive/Recording 10.mp4"

# Check if file exists
if not os.path.exists(AUDIO_FILE):
    print(" File not found!")
    print("Check the exact file path in Google Drive.")
    print("\nFiles in MyDrive:")
    print(os.listdir("/content/drive/MyDrive"))
    raise FileNotFoundError(AUDIO_FILE)

print(" File Found:", AUDIO_FILE)

MODEL_NAME = "superb/wav2vec2-base-superb-er"

print("Loading model...")
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
model = Wav2Vec2ForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()

print("Loading audio...")

speech, sr = torchaudio.load(AUDIO_FILE)

print("Original Sample Rate:", sr)
print("Audio Shape:", speech.shape)

# Convert stereo to mono
if speech.shape[0] > 1:
    speech = speech.mean(dim=0)

speech = speech.squeeze()

# Resample to 16 kHz
if sr != 16000:
    resampler = torchaudio.transforms.Resample(sr, 16000)
    speech = resampler(speech)
    sr = 16000

print("Processed Sample Rate:", sr)

# Feature Extraction
inputs = feature_extractor(
    speech.numpy(),
    sampling_rate=16000,
    return_tensors="pt",
    padding=True
)

# Emotion Prediction
with torch.no_grad():
    logits = model(**inputs).logits

predicted_id = torch.argmax(logits, dim=-1).item()
emotion = model.config.id2label[predicted_id]

print("\n Detected Emotion:", emotion)

# Confidence Scores
probs = F.softmax(logits, dim=-1).squeeze()

print("\n Confidence Scores:")
print("-" * 30)

for i, p in enumerate(probs):
    label = model.config.id2label[i]
    print(f"{label:10s}: {p.item()*100:.2f}%")